# 1. 왜 timing-only dataset을 만들까?

배치를 잘 고르는 모델이 아니라 **기록된 Human 입력 처리 시도 사이의 간격**을 배우기 위한 첫 dataset입니다.
213게임 사용·22게임 격리·11게임 Human 없음 제외는 사용자가 승인했습니다. 승패로 고르지 않습니다.
raw·과거 감사 report는 읽기 전용입니다. 이번에는 dataset 생성만 하고 모델 fit·사이트 적용·engine replay는 하지 않습니다.
명세: `ai-harness/ml-experiments/experiments/human-timing-dataset.md`.


In [ ]:
import hashlib
import json
import math
import os
import platform
from importlib.metadata import version
from collections import Counter
from pathlib import Path

EXPERIMENT_ID = "human-timing-dataset"
ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
DEPENDENCY = ROOT / "scripts/ml/notebooks/human_action_data_readiness.ipynb"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_timing_dataset.ipynb"
shared = {"__name__": "human_timing_shared"}
for cell in json.loads(DEPENDENCY.read_text(encoding="utf-8"))["cells"]:
    if cell["cell_type"] == "code" and "definitions" in cell["metadata"].get("tags", []):
        exec(compile("".join(cell["source"]), str(DEPENDENCY), "exec"), shared)
for name in ("require", "finite", "integer", "reject_constant", "json_bytes", "sha256_file",
             "iter_rows", "normalize_event", "scan_source", "inspect_segment", "interval_summary"):
    globals()[name] = shared[name]

DEFAULT_SOURCE = "data/datasets/tetris-ml-snapshots-backup-0908/raw/game-events.jsonl"
DEFAULT_QUALITY = "data/ml-experiments/human-game-quality-audit/full-report.json"
APPROVED_QUALITY_SHA = "97388649259414fe2f5119e76ffa57922b3202f85c920540674f0fc494973ff4"
APPROVED_POLICY = {"shortGameReviewSeconds": 30, "maxGameDurationSeconds": 600,
                   "abandonmentTailMinPieces": 5, "aiOnlyTailReviewSeconds": 60}
SPLIT_SEED = "human-timing-v1-20261024"
SPLITS = ("train", "validation", "test")


# 2. 승인된 게임과 원본의 연결을 확인합니다

입력은 바꾸지 않습니다. 품질 report는 당시 provisional 표기를 남기고 이 dataset의 manifest에 승인을 별도로 기록합니다.
**정책 승인**과 **과거 report 변조**는 다릅니다. 고정 SHA와 full/원본 연결·분류 목록·합계를 확인하고 맞지 않으면 중단합니다.
PVE Human만, PVP 양쪽 Human을 사용합니다. AI 이벤트는 절대 시간 학습 표본이 아닙니다.


In [ ]:
def validate_quality(quality, source_sha, source_bytes):
    require(isinstance(quality, dict) and quality.get("reportSchemaVersion") == 1
            and quality.get("experimentId") == "human-game-quality-audit", "QUALITY_SCHEMA_MISMATCH")
    require(quality.get("runMode") == "full" and quality.get("fullCohortCovered") is True
            and quality.get("qualityReviewComplete") is True
            and quality.get("sourceContractValid") is True and quality.get("sourceUnchanged") is True,
            "QUALITY_FULL_VALID_REQUIRED")
    require(quality.get("policy") == APPROVED_POLICY, "QUALITY_POLICY_MISMATCH")
    require(quality["source"]["sha256"] == source_sha and quality["source"]["bytes"] == source_bytes,
            "QUALITY_SOURCE_MISMATCH")
    games = quality["games"]
    require(isinstance(games, list) and all(isinstance(g, dict) and isinstance(g.get("gameId"), str) for g in games),
            "QUALITY_GAMES_INVALID")
    ids = [g["gameId"] for g in games]
    require(ids == sorted(set(ids)) and quality["cohort"]["gameIds"] == ids
            and quality["cohort"]["fullSourceGameCount"] == len(ids), "QUALITY_GAME_COHORT_MISMATCH")
    counts = Counter()
    for status in ("include_candidate", "exclude_candidate", "review"):
        selected = [g["gameId"] for g in games if g.get("status") == status]
        require(quality["lists"][status] == selected
                and quality["summary"]["classificationCounts"].get(status, 0) == len(selected),
                "QUALITY_LIST_COUNT_MISMATCH")
        counts[status] = len(selected)
    require(sum(counts.values()) == len(games), "QUALITY_STATUS_INVALID")
    require(all(not g["reasons"] and g["humanPlayers"] and g["mode"] in ("pve", "pvp")
                for g in games if g["status"] == "include_candidate"), "QUALITY_INCLUDE_INVALID")
    return {g["gameId"]: g for g in games}


def split_games(game_ids):
    require(len(game_ids) == len(set(game_ids)) and len(game_ids) >= 3, "THREE_UNIQUE_GAMES_REQUIRED")
    ordered = sorted(game_ids, key=lambda game: (hashlib.sha256((SPLIT_SEED + "\n" + game).encode()).hexdigest(), game))
    train_count = min(max(1, math.floor(.70 * len(ordered))), len(ordered) - 2)
    validation_count = max(1, math.floor(.15 * len(ordered)))
    groups = {"train": sorted(ordered[:train_count]),
              "validation": sorted(ordered[train_count:train_count + validation_count]),
              "test": sorted(ordered[train_count + validation_count:])}
    require(all(groups.values()) and sum(map(len, groups.values())) == len(game_ids), "SPLIT_COUNT_MISMATCH")
    require(not any(set(groups[a]) & set(groups[b]) for i, a in enumerate(SPLITS) for b in SPLITS[i + 1:]),
            "GAME_SPLIT_LEAKAGE")
    return groups


# 3. 손으로 따라 보는 인접 입력 pair

move(1000ms) → rotate(1120ms) → hard_drop(1300ms)이면 label은 120ms·180ms입니다.
첫 행동까지의 반응시간은 모릅니다. feature는 **이전 action 하나**, 뒤 action은 진단 전용입니다.
sequence가 1→3→4라면 1→3은 버리고 3→4만 봅니다. 1→4로 새 pair를 만들지 않습니다.
0ms·긴 간격도 관측값 그대로 두며, hold가 있는 segment 전체는 이전 단계에서 분리합니다.


In [ ]:
def extract_pairs(inputs, meta, player_number):
    rows, rejected = [], []
    for current_index in range(1, len(inputs)):
        previous, current = inputs[current_index - 1], inputs[current_index]
        reasons = []
        if (not integer(previous.get("pieceIndex")) or not integer(current.get("pieceIndex"))
                or previous["pieceIndex"] != current["pieceIndex"] or current["pieceIndex"] != meta["pieceIndex"]):
            reasons.append("PAIR_PIECE_MISMATCH")
        if (not integer(previous.get("sequence")) or not integer(current.get("sequence"))
                or current["sequence"] != previous["sequence"] + 1):
            reasons.append("PAIR_SEQUENCE_NOT_ADJACENT")
        if (not integer(previous.get("tick")) or not integer(current.get("tick"))
                or current["tick"] < previous["tick"]):
            reasons.append("PAIR_TICK_REVERSED_OR_INVALID")
        before, after = previous.get("createdAt"), current.get("createdAt")
        if not (finite(before) and finite(after) and before >= 0 and after >= before):
            reasons.append("PAIR_TIMESTAMP_INVALID")
        lineage = {"gameId": meta["gameId"], "playerNumber": player_number,
                   "pieceIndex": meta["pieceIndex"], "sourceLine": meta["line"],
                   "previousInputIndex": current_index - 1, "currentInputIndex": current_index}
        if reasons:
            rejected.append({**lineage, "reasons": reasons})
            continue
        require(previous.get("action") in shared["ACTIONS"] and current.get("action") in shared["ACTIONS"]
                and "hold" not in (previous["action"], current["action"]), "PAIR_ACTION_CONTRACT")
        delta = after - before
        require(finite(delta) and delta >= 0, "PAIR_INTERVAL_NONFINITE")
        sample_id = hashlib.sha256(json_bytes(lineage)).hexdigest()
        rows.append({"schemaVersion": 1, "sampleId": sample_id, **lineage,
                     "previousAction": previous["action"], "nextActionDiagnostic": current["action"],
                     "recordedIntervalMs": delta})
    return rows, rejected


In [ ]:
example = [{"action": action, "pieceIndex": 1, "sequence": i + 1, "tick": i,
            "createdAt": timestamp} for i, (action, timestamp) in enumerate(
            (("move_left", 1000), ("rotate", 1120), ("hard_drop", 1300)))]
example_rows, example_rejected = extract_pairs(example, {"gameId": "synthetic", "pieceIndex": 1, "line": 1}, 1)
assert [row["recordedIntervalMs"] for row in example_rows] == [120, 180]
assert not example_rejected
print("Synthetic only:", [(row["previousAction"], row["recordedIntervalMs"]) for row in example_rows])


# 4. 게임 → Human → 구간 → pair 순서로 정제합니다

기존 parser의 identity·구간 판정을 재사용합니다. 다른 piece의 앞뒤 기록을 연결하지 않습니다.
한 segment의 시간 구조가 usable이어도 개별 pair의 sequence +1·tick 비역전 검사는 추가로 필요합니다.
piece gap은 한 segment 안의 정상 pair를 자동으로 버리는 이유가 아니지만 공백을 가로질러 표본을 만들지는 않습니다.
v3/40행·다른 ruleset은 이번 legacy dataset에 섞지 않습니다.


In [ ]:
def build_timing_dataset(source, quality, mode="smoke"):
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    source = Path(source)
    source_sha = sha256_file(source)
    quality_games = validate_quality(quality, source_sha, source.stat().st_size)
    index = scan_source(source)
    require(not index["invalidReasons"] and not any(index["counts"].get(key, 0)
            for key in ("parse_failure", "identityConflictRows", "invalid")), "SOURCE_CONTRACT_INVALID")
    require(set(index["games"]) == set(quality_games), "QUALITY_SOURCE_GAMES_MISMATCH")
    eligible_games = quality["lists"]["include_candidate"]
    selected = eligible_games[:5] if mode == "smoke" else eligible_games[:]
    groups = split_games(selected)
    game_split = {game: split for split, games in groups.items() for game in games}
    rows, rejected_segments, rejected_pairs = [], [], []
    counts, segment_reasons, pair_reasons = Counter(), Counter(), Counter()
    metadata_counts = {key: Counter() for key in ("geometry", "ruleset", "schemaVersion", "trust", "stage")}
    strata = {}
    for line, raw, kind in iter_rows(source):
        meta = index["validByLine"].get(line)
        if meta is None or meta["gameId"] not in game_split:
            continue
        require(kind == "parsed", "SOURCE_CHANGED_DURING_DATASET")
        _, _, payload, _ = normalize_event(raw)
        counts["humanSegments"] += 1
        for key in metadata_counts:
            metadata_counts[key][str(meta[key])] += 1
        group = (quality_games[meta["gameId"]]["mode"], quality_games[meta["gameId"]]["difficulty"])
        stratum = strata.setdefault(group, Counter())
        stratum["humanSegments"] += 1
        item = inspect_segment(payload, meta, index["problems"].get(line, ()))
        reason = None
        if meta["geometry"] != "20x10" or meta["ruleset"] != "unspecified" or meta["schemaVersion"] not in (1, 2):
            reason = "UNSUPPORTED_LOG_CONTRACT"
        elif item["counts"].get("holdAttempts", 0):
            reason = "HOLD_SEGMENT"
        elif item["readiness"]["logged_inter_action_timing"] != "usable":
            reason = "TIMING_" + item["readiness"]["logged_inter_action_timing"].upper()
        lineage = {"gameId": meta["gameId"], "playerNumber": index["pseudonyms"][meta["gameId"]][meta["playerId"]],
                   "pieceIndex": meta["pieceIndex"], "sourceLine": line}
        if reason is not None:
            segment_reasons[reason] += 1
            stratum["rejectedSegments"] += 1
            rejected_segments.append({**lineage, "reason": reason})
            continue
        counts["eligibleSegments"] += 1
        stratum["eligibleSegments"] += 1
        counts["pieceGapSegments"] += int("PIECE_GAP" in item["reasons"])
        pairs, rejected = extract_pairs(payload["inputSequence"], meta, lineage["playerNumber"])
        counts["candidatePairs"] += len(payload["inputSequence"]) - 1
        counts["representedSegments"] += bool(pairs)
        for row in pairs:
            row["split"] = game_split[meta["gameId"]]
        rows.extend(pairs)
        stratum["pairs"] += len(pairs)
        rejected_pairs.extend(rejected)
        for row in rejected:
            pair_reasons.update(row["reasons"])
    require(sha256_file(source) == source_sha, "SOURCE_CHECKSUM_CHANGED")
    expected = sum(p["humanSegments"] for game in selected for p in quality_games[game]["humanPlayers"])
    require(counts["humanSegments"] == expected
            and counts["humanSegments"] == counts["eligibleSegments"] + len(rejected_segments),
            "SEGMENT_COUNT_MISMATCH")
    rows.sort(key=lambda r: (r["gameId"], r["playerNumber"], r["pieceIndex"], r["sourceLine"], r["currentInputIndex"]))
    for rejected in (rejected_segments, rejected_pairs):
        rejected.sort(key=lambda r: (r["gameId"], r["playerNumber"], r["pieceIndex"], r["sourceLine"], r.get("currentInputIndex", -1)))
    require(len({r["sampleId"] for r in rows}) == len(rows), "DUPLICATE_SAMPLE")
    require(counts["candidatePairs"] == len(rows) + len(rejected_pairs), "PAIR_COUNT_MISMATCH")
    split_stats = {}
    for split in SPLITS:
        samples = [r for r in rows if r["split"] == split]
        require(samples, "SPLIT_WITHOUT_PAIRS:" + split)
        split_stats[split] = {"games": len(groups[split]), "representedGames": len({r["gameId"] for r in samples}),
                              "pairs": len(samples), "zeroMsPairs": sum(r["recordedIntervalMs"] == 0 for r in samples),
                              "zeroMsFraction": sum(r["recordedIntervalMs"] == 0 for r in samples) / len(samples),
                              "intervalMs": interval_summary([r["recordedIntervalMs"] for r in samples]),
                              "previousActionCounts": dict(sorted(Counter(r["previousAction"] for r in samples).items()))}
    counts["pairs"] = len(rows)
    counts["rejectedSegments"] = len(rejected_segments)
    counts["rejectedPairs"] = len(rejected_pairs)
    counts["pairlessSegments"] = counts["eligibleSegments"] - counts["representedSegments"]
    manifest = {"datasetSchemaVersion": 1, "experimentId": EXPERIMENT_ID,
        "status": "timing_dataset_smoke_valid" if mode == "smoke" else "timing_dataset_generated",
        "runMode": mode, "fullCohortCovered": mode == "full", "dataDecisionEligible": mode == "full",
        "policyDecisionEligible": False, "canonicalResultEligible": False, "modelFitCount": 0,
        "environment": {"python": platform.python_version(), "pandas": version("pandas"), "matplotlib": version("matplotlib")},
        "featureColumns": ["previousAction"], "labelColumn": "recordedIntervalMs",
        "diagnosticOnlyColumns": ["nextActionDiagnostic", "gameId", "playerNumber", "pieceIndex", "sourceLine",
                                  "previousInputIndex", "currentInputIndex", "split", "sampleId"],
        "timeMeaning": "recorded_runtime_processing_attempt_interval_not_keypress_or_reaction",
        "supportedLogContract": {"geometry": "20x10", "ruleset": "unspecified", "schemaVersions": [1, 2],
                                 "inputOutcome": "unverified", "historicalBuildBinding": "unverified"},
        "source": {"sha256": source_sha, "bytes": source.stat().st_size},
        "qualityPolicy": {"id": "quality-policy-v1", "approval": "user_approved_2026-10-24",
                          "parameters": dict(APPROVED_POLICY),
                          "reportSha256": hashlib.sha256(json_bytes(quality)).hexdigest(),
                          "gameLists": quality["lists"]},
        "cohort": {"selectedGameIds": selected, "selection": "first_five_include_games" if mode == "smoke" else "all_include_games",
                   "qualityEligibleGames": len(eligible_games)},
        "split": {"method": "sha256_seed_game_order_70_15_remainder", "seed": SPLIT_SEED, "gameIds": groups,
                  "gameIdLeakage": 0, "userHoldout": False, "temporalHoldout": False},
        "counts": dict(counts), "segmentRejectionReasons": dict(sorted(segment_reasons.items())),
        "pairRejectionReasons": dict(sorted(pair_reasons.items())), "splitStatistics": split_stats,
        "observedHumanSegmentMetadata": {key: dict(sorted(value.items())) for key, value in metadata_counts.items()},
        "byModeDifficulty": [{"mode": key[0], "difficulty": key[1], **dict(value)} for key, value in sorted(strata.items())],
        "limits": ["no_model_fit", "no_replay", "no_hold_policy", "no_first_reaction_time",
                   "no_personalization_claim", "no_v3_transfer", "validation_test_statistics_diagnostic_only"]}
    return rows, rejected_segments, rejected_pairs, manifest


# 5. 게임을 나눈 뒤에는 표본을 다른 split으로 옮기지 않습니다

고정 seed + gameId의 SHA-256 정렬로 70%/15%/나머지를 나눕니다. 213게임이면 149/31/33게임입니다.
PVP 양쪽 player도 같은 game이므로 함께 이동합니다. label·승패·평가 결과로 split을 고르지 않습니다.
분포 표는 누락·편향 확인용입니다. train/validation/test 전체에서 분포를 fit하지 않으며 이번에는 모델 자체가 없습니다.


# 6. 새 폴더에만 기록합니다

JSONL은 한 행씩 쓰고 파일 SHA·bytes를 기록합니다. raw 전체나 JSONL 전체를 하나의 문자열로 합치지 않습니다.
기존 폴더·symlink·source 경로는 덮지 않습니다. manifest를 마지막에 쓰며 manifest 없는 부분 폴더는 완료본이 아닙니다.
실패해 남은 부분 폴더를 자동 삭제하거나 재사용하지 않고 새 version 경로로 재시도합니다.


In [ ]:
def checked_dataset_output(root, output_name, mode):
    root = Path(root).resolve()
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    output = Path(output_name)
    require(".." not in output.parts, "OUTPUT_PARENT_ALIAS_NOT_ALLOWED")
    output = output if output.is_absolute() else root / output
    require(not output.exists() and not output.is_symlink(), "OUTPUT_ALREADY_EXISTS_OR_SYMLINK")
    parent = output.parent.resolve(strict=True)
    canonical = parent / output.name
    if mode == "full":
        require(canonical.is_relative_to(root) and parent == (root / "data/datasets").resolve()
                and output.name.startswith("human-timing-v1-"), "FULL_DATASET_LOCATION_NOT_ALLOWED")
    else:
        temporary = (Path("/private/tmp").resolve(), Path(shared["tempfile"].gettempdir()).resolve())
        evidence = root / "data/ml-experiments" / EXPERIMENT_ID
        require(canonical.is_relative_to(evidence) or any(canonical.is_relative_to(p) for p in temporary),
                "SMOKE_DATASET_LOCATION_NOT_ALLOWED")
    return canonical


def publish_dataset(output, rows, rejected_segments, rejected_pairs, manifest):
    output = Path(output)
    output.mkdir()
    files = {}
    # JSONL 전체를 하나의 문자열로 합치지 않고 한 행씩 기록한다.
    for name in (*[split + ".jsonl" for split in SPLITS], "rejected-segments.jsonl", "rejected-pairs.jsonl"):
        records = (r for r in rows if r["split"] == name[:-6]) if name[:-6] in SPLITS else (
            rejected_segments if name == "rejected-segments.jsonl" else rejected_pairs)
        digest, count, size = hashlib.sha256(), 0, 0
        with (output / name).open("xb") as handle:
            for record in records:
                data = (json.dumps(record, ensure_ascii=False, sort_keys=True, allow_nan=False, separators=(",", ":")) + "\n").encode()
                handle.write(data)
                digest.update(data)
                count += 1
                size += len(data)
        files[name] = {"sha256": digest.hexdigest(), "rows": count, "bytes": size}
    published = {**manifest, "files": files}
    with (output / "manifest.json").open("xb") as handle:
        handle.write(json_bytes(published))
    return published


def run_timing_from_environment():
    mode = os.environ.get("HUMAN_TIMING_RUN_MODE", "smoke")
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    require(mode != "full" or "HUMAN_TIMING_OUTPUT" in os.environ, "FULL_OUTPUT_PATH_REQUIRED")
    def input_path(name):
        relative = Path(name)
        require(not relative.is_absolute() and ".." not in relative.parts, "INPUT_MUST_BE_REPO_RELATIVE")
        path = (ROOT / relative).resolve(strict=True)
        require(path.is_relative_to(ROOT) and path.is_file(), "INPUT_OUTSIDE_REPOSITORY")
        return path
    source = input_path(os.environ.get("HUMAN_TIMING_SOURCE", DEFAULT_SOURCE))
    quality_path = input_path(os.environ.get("HUMAN_TIMING_QUALITY_REPORT", DEFAULT_QUALITY))
    output = checked_dataset_output(ROOT, os.environ.get("HUMAN_TIMING_OUTPUT", "/private/tmp/human-timing-dataset-smoke"), mode)
    quality_sha = sha256_file(quality_path)
    require(quality_sha == APPROVED_QUALITY_SHA, "APPROVED_QUALITY_SHA_MISMATCH")
    quality = json.loads(quality_path.read_text(encoding="utf-8"), parse_constant=reject_constant)
    require(hashlib.sha256(json_bytes(quality)).hexdigest() == quality_sha, "QUALITY_CANONICAL_BYTES_MISMATCH")
    code_before = {"sourceNotebookSha256": sha256_file(SOURCE_NOTEBOOK), "dependencyNotebookSha256": sha256_file(DEPENDENCY)}
    rows, rejected_segments, rejected_pairs, manifest = build_timing_dataset(source, quality, mode)
    require(sha256_file(quality_path) == quality_sha, "QUALITY_CHANGED_DURING_DATASET")
    require(code_before == {"sourceNotebookSha256": sha256_file(SOURCE_NOTEBOOK),
                            "dependencyNotebookSha256": sha256_file(DEPENDENCY)}, "CODE_CHANGED_DURING_DATASET")
    manifest["code"] = code_before
    manifest["source"]["path"] = source.relative_to(ROOT).as_posix()
    manifest = publish_dataset(output, rows, rejected_segments, rejected_pairs, manifest)
    return rows, manifest


# 7. smoke부터 실행합니다

기본 smoke는 승인된 include 목록의 앞 5게임만 처리합니다. full은 명세의 명령으로 사용자가 실행합니다.
full이라는 출력 이름만 붙여서는 full이 되지 않습니다. output dataset 폴더를 미리 만들면 보호 검사에서 중단됩니다.


In [ ]:
rows, manifest = run_timing_from_environment()
print(json.dumps({key: manifest[key] for key in ("status", "runMode", "counts", "splitStatistics")}, ensure_ascii=False, indent=2))


# 8. pandas 표로 표본 수와 제외 이유를 읽습니다

게임 수·구간 수·pair 수는 다른 분모입니다. 동일 game의 여러 player·piece를 서로 독립인 게임으로 세지 않습니다.
간격 0ms는 관측된 runtime batch 효과일 수 있습니다. 긴 간격도 임의 제거하지 않았습니다.
validation/test의 표는 품질 진단일 뿐 학습용 분포가 아닙니다.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pair_table = pd.DataFrame(rows)
split_table = pd.DataFrame(manifest["splitStatistics"]).T[["games", "representedGames", "pairs", "zeroMsPairs"]]
split_table["zeroMsPercent"] = (split_table["zeroMsPairs"] / split_table["pairs"] * 100).round(2)
rejection_table = pd.Series(manifest["segmentRejectionReasons"], dtype="int64", name="segments").to_frame()
action_table = pd.crosstab(pair_table["previousAction"], pair_table["split"]).reindex(columns=SPLITS, fill_value=0)
print(f"Mode: {manifest['runMode']} | quality-policy-v1 | feature=previousAction | label=recordedIntervalMs")
display(split_table)
display(rejection_table)
display(pd.DataFrame(manifest["byModeDifficulty"]).fillna(0))
display(action_table)
display(pair_table[["previousAction", "nextActionDiagnostic", "recordedIntervalMs", "split"]].head(10))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), layout="constrained")
split_table["pairs"].plot.bar(ax=axes[0], rot=0, color="#0072B2")
axes[0].set(title="Pairs by game split", xlabel="Split", ylabel="Adjacent pairs")
# 원본 label은 그대로 두고 표시만 log1p로 바꿔 0ms도 보존한다.
axes[1].hist([math.log1p(value) for value in pair_table["recordedIntervalMs"]], bins=30, color="#0072B2", edgecolor="white")
axes[1].set(title="Recorded intervals (all splits; diagnostic)", xlabel="log1p(milliseconds), zero retained", ylabel="Pairs")
if rejection_table.empty:
    axes[2].text(.5, .5, "No rejected segments", ha="center", transform=axes[2].transAxes)
else:
    rejection_table.plot.barh(ax=axes[2], legend=False, color="#E69F00")
axes[2].set(title="Segment rejection reasons", xlabel="Segments", ylabel="Reason")
fig.suptitle(f"Human timing dataset | {manifest['runMode']} | no model fit")
plt.show()
plt.close(fig)


# 9. 누수·원본 불변·결정성을 확인합니다

각 sampleId는 하나뿐이고 game은 정확히 한 split에 있습니다. 한 구간의 i-1/i 이외 pair는 만들지 않습니다.
원본 SHA·품질 report SHA·Notebook SHA가 실행 중 바뀌면 게시하지 않습니다. JSONL 파일별 SHA를 manifest로 대조합니다.
source output은 비워 보관하고 실행본만 ignored 경로에 둡니다.


# 10. 이 dataset으로 아직 말할 수 없는 것

기록된 처리 시도 간격이지 실제 키 누름 간격·반응시간·성공 입력 간격이 아닙니다.
개인화·상태 조건부 행동·hold·강한 공격 AI·v3/40행 적용의 증거가 아닙니다.
다음 gate는 사용자 full 생성 결과의 count·분포·제외 편향 확인입니다. 그 뒤 별도 실험에서 train-only 시간 분포 baseline을 비교합니다.
